# Asset Universe Screening: 1,000 → 10 Optimal Portfolio
## Regime-Switching Risk-Parity Crypto Index Vault

Systematic quantitative screening pipeline:
1. Fetch top 1,000 crypto assets by market cap
2. Apply liquidity and statistical quality filters
3. Compute risk-return profiles for each surviving asset
4. Hierarchical correlation clustering
5. Diversification-optimized selection of final 10
6. Validation via spanning tests and efficiency metrics

**Run via:** VS Code → Select Kernel → Colab → T4 GPU  
**Runtime:** ~30–45 minutes (data fetching is the bottleneck)

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Install dependencies and clone the project repo onto the Colab runtime.
# When using the VS Code Colab extension, the notebook file is local
# but code executes on Google's remote runtime.

!pip install -q ccxt numpy pandas scipy scikit-learn statsmodels matplotlib seaborn networkx tqdm pyyaml

import os

REPO_URL = "https://github.com/abailey81/Regime-Switching-Risk-Parity-Crypto-Index-Vault.git"
PROJECT_DIR = "/content/Regime-Switching-Risk-Parity-Crypto-Index-Vault"

if not os.path.exists(PROJECT_DIR):
    print("Cloning repository from GitHub...")
    !git clone {REPO_URL}
else:
    print("Repository already cloned. Pulling latest...")
    !cd {PROJECT_DIR} && git pull --ff-only

os.chdir(os.path.join(PROJECT_DIR, "ml"))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Contents: {sorted(os.listdir('.'))}")

In [ ]:
# ============================================================
# Cell 2: Initialize Screener
# ============================================================

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

from analysis.universe_screening import UniverseScreener

# ---- Chart style ----
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
})
PALETTE = sns.color_palette("husl", 12)
SAVE_DIR = "results/screening"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Load config ----
with open("config.yaml") as f:
    config = yaml.safe_load(f)

screener = UniverseScreener(config=config)
print("UniverseScreener initialized.")
print(f"Config assets (current): {config['data']['assets']}")
print(f"Data source: {config['data']['source']}")
print(f"Fallback exchanges: {config['data']['fallback_exchanges']}")

---
## Stage 1: Fetch Top 1,000 by Market Cap

Pull the universe of crypto assets ranked by market capitalisation from exchange data. This gives us the broadest feasible starting point before any filters are applied.

In [ ]:
# ============================================================
# Cell 3: Stage 1 --- Fetch Universe
# ============================================================

universe_df = screener.fetch_universe(top_n=1000)

print(f"Universe fetched: {len(universe_df)} assets")
print(f"\nTop 20 by market cap:")
display(universe_df.head(20))

# ---- Distribution plots ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Market cap distribution (log scale)
ax = axes[0]
mcap = universe_df['market_cap'] if 'market_cap' in universe_df.columns else None
if mcap is not None and len(mcap) > 0:
    ax.hist(np.log10(mcap.clip(lower=1)), bins=40, color=PALETTE[0], edgecolor='white', alpha=0.85)
    ax.set_xlabel('log10(Market Cap USD)')
    ax.set_ylabel('Count')
    ax.set_title('Market Cap Distribution')

# (b) 24h volume distribution (log scale)
ax = axes[1]
vol24 = universe_df['volume_24h'] if 'volume_24h' in universe_df.columns else None
if vol24 is not None and len(vol24) > 0:
    ax.hist(np.log10(vol24.clip(lower=1)), bins=40, color=PALETTE[1], edgecolor='white', alpha=0.85)
    ax.set_xlabel('log10(24h Volume USD)')
    ax.set_ylabel('Count')
    ax.set_title('24h Volume Distribution')

# (c) Scatter: market cap vs volume
ax = axes[2]
if mcap is not None and vol24 is not None:
    ax.scatter(
        np.log10(mcap.clip(lower=1)),
        np.log10(vol24.clip(lower=1)),
        alpha=0.4, s=15, color=PALETTE[2], edgecolors='none',
    )
    ax.set_xlabel('log10(Market Cap USD)')
    ax.set_ylabel('log10(24h Volume USD)')
    ax.set_title('Market Cap vs Volume')
    # Label top 5
    for _, row in universe_df.head(5).iterrows():
        symbol = row.get('symbol', row.name if isinstance(row.name, str) else '')
        mc_val = np.log10(max(row.get('market_cap', 1), 1))
        v_val = np.log10(max(row.get('volume_24h', 1), 1))
        ax.annotate(symbol, (mc_val, v_val), fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage1_universe_distributions.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage1_universe_distributions.png")

---
## Stage 2: Fetch Historical OHLCV Data

Retrieve 2 years of daily OHLCV data for all assets in the universe. Data coverage determines which assets can be analysed reliably.

In [ ]:
# ============================================================
# Cell 4: Stage 2 --- Fetch OHLCV Data
# ============================================================

symbols = universe_df['symbol'].tolist() if 'symbol' in universe_df.columns else universe_df.index.tolist()

print(f"Fetching data for {len(symbols)} assets (lookback=730 days)...")
print("This is the slowest step --- expect 15-25 minutes.")

ohlcv_data = screener.fetch_universe_data(symbols=symbols, lookback_days=730)

print(f"\nData received for {len(ohlcv_data)} assets.")

# ---- Coverage summary ----
coverage = []
for sym, df in ohlcv_data.items():
    n_rows = len(df)
    date_range = (df.index.min(), df.index.max()) if n_rows > 0 else (None, None)
    coverage.append({
        'symbol': sym,
        'rows': n_rows,
        'start': date_range[0],
        'end': date_range[1],
        'coverage_pct': round(n_rows / 730 * 100, 1) if n_rows > 0 else 0.0,
    })

coverage_df = pd.DataFrame(coverage).sort_values('rows', ascending=False)
print(f"\nCoverage summary:")
print(f"  Assets with >= 90% coverage: {(coverage_df['coverage_pct'] >= 90).sum()}")
print(f"  Assets with >= 50% coverage: {(coverage_df['coverage_pct'] >= 50).sum()}")
print(f"  Assets with < 50% coverage:  {(coverage_df['coverage_pct'] < 50).sum()}")

# ---- Heatmap of data availability ----
# Build a binary matrix: assets x time (monthly resolution for readability)
all_dates = pd.date_range(
    start=min(df.index.min() for df in ohlcv_data.values() if len(df) > 0),
    end=max(df.index.max() for df in ohlcv_data.values() if len(df) > 0),
    freq='W',
)

# Take top 80 by coverage for readability
top_syms = coverage_df.head(80)['symbol'].tolist()
avail_matrix = pd.DataFrame(0, index=top_syms, columns=all_dates)
for sym in top_syms:
    if sym in ohlcv_data and len(ohlcv_data[sym]) > 0:
        sym_dates = ohlcv_data[sym].index
        for week in all_dates:
            week_start = week - pd.Timedelta(days=3)
            week_end = week + pd.Timedelta(days=3)
            if ((sym_dates >= week_start) & (sym_dates <= week_end)).any():
                avail_matrix.loc[sym, week] = 1

fig, ax = plt.subplots(figsize=(18, 12))
sns.heatmap(
    avail_matrix.values,
    cmap=['#f0f0f0', PALETTE[0]],
    cbar=False, ax=ax,
    yticklabels=top_syms,
)
n_cols = len(all_dates)
tick_positions = np.linspace(0, n_cols - 1, min(12, n_cols)).astype(int)
ax.set_xticks(tick_positions)
ax.set_xticklabels([all_dates[i].strftime('%Y-%m') for i in tick_positions], rotation=45, ha='right')
ax.set_title('Data Availability Heatmap (Top 80 Assets by Coverage)', fontweight='bold')
ax.set_ylabel('Asset')
ax.set_xlabel('Date')
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage2_data_availability.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage2_data_availability.png")

---
## Stage 3: Liquidity Filter

Remove illiquid assets that cannot be traded at scale. Filters on minimum daily USD volume and minimum history length. This is the single largest reducer of the universe.

In [ ]:
# ============================================================
# Cell 5: Stage 3 --- Liquidity Filter
# ============================================================

pre_liquidity_count = len(symbols)
liquid_assets = screener.filter_liquidity()
post_liquidity_count = len(liquid_assets)

removed_count = pre_liquidity_count - post_liquidity_count

print(f"Liquidity filter results:")
print(f"  Before: {pre_liquidity_count}")
print(f"  After:  {post_liquidity_count}")
print(f"  Removed: {removed_count}")
print(f"\nSurviving assets (first 30): {liquid_assets[:30]}")

# ---- Funnel / waterfall chart ----
stages = ['Initial Universe', 'After Liquidity Filter']
counts = [pre_liquidity_count, post_liquidity_count]
colors = [PALETTE[0], PALETTE[2]]

fig, ax = plt.subplots(figsize=(10, 6))

# Draw as a horizontal funnel
bar_heights = [0.8, 0.8]
y_positions = [1, 0]

bars = ax.barh(y_positions, counts, height=bar_heights, color=colors, edgecolor='white', linewidth=2)

for bar, count, stage in zip(bars, counts, stages):
    ax.text(
        bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
        f'{count} assets', va='center', fontweight='bold', fontsize=13,
    )

# Arrow showing reduction
ax.annotate(
    f'-{removed_count} removed\n({removed_count/pre_liquidity_count*100:.0f}%)',
    xy=(post_liquidity_count / 2, 0.5),
    fontsize=11, ha='center', va='center',
    color='#cc0000', fontweight='bold',
)

ax.set_yticks(y_positions)
ax.set_yticklabels(stages, fontsize=12)
ax.set_xlabel('Number of Assets')
ax.set_title('Liquidity Filter Funnel: 500 -> N', fontweight='bold')
ax.set_xlim(0, pre_liquidity_count * 1.15)
sns.despine(left=True)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage3_liquidity_funnel.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage3_liquidity_funnel.png")

---
## Stage 4: Statistical Quality Filter

Remove assets with poor statistical properties: non-stationary returns (ADF test failure), degenerate distributions, or structural breaks that would undermine model reliability.

In [ ]:
# ============================================================
# Cell 6: Stage 4 --- Statistical Quality Filter
# ============================================================

pre_stat_count = len(liquid_assets)
stat_quality_assets = screener.filter_statistical_quality()
post_stat_count = len(stat_quality_assets)

removed_stat = pre_stat_count - post_stat_count

print(f"Statistical quality filter results:")
print(f"  Before: {pre_stat_count}")
print(f"  After:  {post_stat_count}")
print(f"  Removed: {removed_stat}")
print(f"\nSurviving assets (first 30): {stat_quality_assets[:30]}")

# ---- Cumulative funnel chart (all stages so far) ----
funnel_stages = ['Initial (500)', 'Liquidity Filter', 'Statistical Quality']
funnel_counts = [pre_liquidity_count, post_liquidity_count, post_stat_count]
funnel_colors = [PALETTE[0], PALETTE[2], PALETTE[4]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Funnel bar chart
ax = axes[0]
y_pos = list(range(len(funnel_stages) - 1, -1, -1))
bars = ax.barh(y_pos, funnel_counts, color=funnel_colors, edgecolor='white', height=0.7)
for bar, count, stage in zip(bars, funnel_counts, funnel_stages):
    ax.text(
        bar.get_width() + 3, bar.get_y() + bar.get_height() / 2,
        f'{count}', va='center', fontweight='bold', fontsize=12,
    )
ax.set_yticks(y_pos)
ax.set_yticklabels(funnel_stages[::-1], fontsize=11)
ax.set_xlabel('Number of Assets')
ax.set_title('Screening Funnel', fontweight='bold')
ax.set_xlim(0, pre_liquidity_count * 1.15)
sns.despine(left=True, ax=ax)

# (b) Hurst exponent distribution
# Compute Hurst exponents for the surviving assets to show distributional quality
ax = axes[1]
hurst_values = []
hurst_labels = []
for sym in stat_quality_assets[:80]:  # Cap for plotting speed
    if sym in ohlcv_data and len(ohlcv_data[sym]) > 100:
        rets = ohlcv_data[sym]['close'].pct_change().dropna()
        if len(rets) > 100:
            # Simplified rescaled range Hurst estimator
            n = len(rets)
            max_k = min(n // 4, 256)
            if max_k >= 8:
                rs_list = []
                k_list = []
                for k in [8, 16, 32, 64, 128, 256]:
                    if k > max_k:
                        break
                    n_blocks = n // k
                    if n_blocks < 1:
                        break
                    rs_vals = []
                    for b in range(n_blocks):
                        block = rets.values[b * k:(b + 1) * k]
                        mean_b = block.mean()
                        cumdev = np.cumsum(block - mean_b)
                        R = cumdev.max() - cumdev.min()
                        S = block.std(ddof=1)
                        if S > 1e-12:
                            rs_vals.append(R / S)
                    if rs_vals:
                        rs_list.append(np.log(np.mean(rs_vals)))
                        k_list.append(np.log(k))
                if len(k_list) >= 3:
                    slope = np.polyfit(k_list, rs_list, 1)[0]
                    hurst_values.append(slope)
                    hurst_labels.append(sym)

if hurst_values:
    ax.hist(hurst_values, bins=30, color=PALETTE[4], edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Random Walk (H=0.5)')
    ax.set_xlabel('Hurst Exponent')
    ax.set_ylabel('Count')
    ax.set_title('Hurst Exponent Distribution (Surviving Assets)', fontweight='bold')
    ax.legend(fontsize=10)
    print(f"\nHurst exponent stats: mean={np.mean(hurst_values):.3f}, "
          f"median={np.median(hurst_values):.3f}, std={np.std(hurst_values):.3f}")
    print(f"Assets with H < 0.5 (mean-reverting): {sum(1 for h in hurst_values if h < 0.5)}")
    print(f"Assets with H > 0.5 (trending):       {sum(1 for h in hurst_values if h > 0.5)}")
else:
    ax.text(0.5, 0.5, 'Hurst exponents not computable', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage4_statistical_quality.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage4_statistical_quality.png")

---
## Stage 5: Risk-Return Profiles

For each surviving asset, compute a comprehensive profile: annualised return, volatility, Sharpe ratio, maximum drawdown, skewness, kurtosis, idiosyncratic volatility, and Amihud illiquidity ratio.

In [ ]:
# ============================================================
# Cell 7: Stage 5 --- Risk-Return Profiles
# ============================================================

profiles_df = screener.compute_asset_profiles()

print(f"Profiles computed for {len(profiles_df)} assets.")
print(f"\nTop 30 by Sharpe ratio:")

# Sort by Sharpe and display
sharpe_col = 'sharpe_ratio' if 'sharpe_ratio' in profiles_df.columns else 'sharpe'
profiles_sorted = profiles_df.sort_values(sharpe_col, ascending=False)
display_cols = [c for c in ['symbol', 'ann_return', 'ann_volatility', sharpe_col,
                             'max_drawdown', 'skewness', 'kurtosis', 'idio_vol'] if c in profiles_df.columns]
display(profiles_sorted[display_cols].head(30))

# ---- Chart 1: Return vs Volatility (efficient frontier shape) ----
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
ret_col = 'ann_return' if 'ann_return' in profiles_df.columns else 'return'
vol_col = 'ann_volatility' if 'ann_volatility' in profiles_df.columns else 'volatility'

# Color by asset class if available
if 'asset_class' in profiles_df.columns:
    classes = profiles_df['asset_class'].unique()
    class_colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(classes)}
    for cls in classes:
        mask = profiles_df['asset_class'] == cls
        ax.scatter(
            profiles_df.loc[mask, vol_col] * 100,
            profiles_df.loc[mask, ret_col] * 100,
            alpha=0.6, s=30, label=cls, color=class_colors[cls], edgecolors='none',
        )
    ax.legend(fontsize=9, loc='upper left')
else:
    sc = ax.scatter(
        profiles_df[vol_col] * 100,
        profiles_df[ret_col] * 100,
        alpha=0.5, s=25, c=profiles_df[sharpe_col],
        cmap='RdYlGn', edgecolors='none',
    )
    plt.colorbar(sc, ax=ax, label='Sharpe Ratio', shrink=0.8)

# Label top 10 by Sharpe
sym_col = 'symbol' if 'symbol' in profiles_df.columns else profiles_df.index.name or 'index'
for _, row in profiles_sorted.head(10).iterrows():
    sym = row.get('symbol', row.name)
    ax.annotate(
        sym, (row[vol_col] * 100, row[ret_col] * 100),
        fontsize=7, fontweight='bold', alpha=0.8,
    )

ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
ax.set_title('Risk-Return Space', fontweight='bold')
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

# ---- Chart 2: Sharpe vs Max Drawdown ----
ax = axes[1]
dd_col = 'max_drawdown' if 'max_drawdown' in profiles_df.columns else 'drawdown'
ax.scatter(
    profiles_df[dd_col].abs() * 100,
    profiles_df[sharpe_col],
    alpha=0.5, s=25, color=PALETTE[3], edgecolors='none',
)
for _, row in profiles_sorted.head(10).iterrows():
    sym = row.get('symbol', row.name)
    ax.annotate(
        sym, (abs(row[dd_col]) * 100, row[sharpe_col]),
        fontsize=7, fontweight='bold', alpha=0.8,
    )
ax.set_xlabel('Max Drawdown (%)')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe vs Max Drawdown', fontweight='bold')
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

# ---- Chart 3: Top 20 by idiosyncratic volatility ----
ax = axes[2]
if 'idio_vol' in profiles_df.columns:
    idio_sorted = profiles_df.nlargest(20, 'idio_vol')
    syms = [str(row.get('symbol', row.name)) for _, row in idio_sorted.iterrows()]
    vals = idio_sorted['idio_vol'].values * 100
    y_pos = range(len(syms) - 1, -1, -1)
    ax.barh(list(y_pos), vals, color=PALETTE[5], edgecolor='white')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(syms, fontsize=9)
    ax.set_xlabel('Idiosyncratic Volatility (%)')
    ax.set_title('Top 20 by Unique Risk (Diversification Benefit)', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Idiosyncratic volatility not available', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage5_risk_return_profiles.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage5_risk_return_profiles.png")

---
## Stage 6: Correlation Clustering

Hierarchical clustering on the correlation matrix to identify groups of assets that move together. The goal is to select one representative from each cluster to maximise diversification.

In [ ]:
# ============================================================
# Cell 8: Stage 6 --- Correlation Clustering
# ============================================================

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

cluster_df = screener.cluster_assets(n_clusters=20)

print(f"Clustering complete. {len(cluster_df)} assets in {cluster_df['cluster'].nunique()} clusters.")
print(f"\nCluster sizes:")
cluster_sizes = cluster_df['cluster'].value_counts().sort_index()
for cluster_id, size in cluster_sizes.items():
    members = cluster_df[cluster_df['cluster'] == cluster_id]
    sym_col = 'symbol' if 'symbol' in members.columns else members.index.name or 'index'
    member_syms = members['symbol'].tolist() if 'symbol' in members.columns else members.index.tolist()
    preview = ', '.join(str(s) for s in member_syms[:5])
    suffix = f', ... (+{size - 5})' if size > 5 else ''
    print(f"  Cluster {cluster_id:2d} ({size:3d} assets): {preview}{suffix}")

# ---- Chart 1: Dendrogram ----
fig, axes = plt.subplots(2, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [2, 1]})

# Build correlation distance matrix from available return data
# Use cluster_df symbols to build returns matrix
cluster_syms = cluster_df['symbol'].tolist() if 'symbol' in cluster_df.columns else cluster_df.index.tolist()
available_syms = [s for s in cluster_syms if s in ohlcv_data and len(ohlcv_data[s]) > 60]

# Build returns matrix
returns_dict = {}
for sym in available_syms:
    rets = ohlcv_data[sym]['close'].pct_change().dropna()
    returns_dict[sym] = rets

if returns_dict:
    ret_matrix = pd.DataFrame(returns_dict).dropna(how='all').fillna(0)
    # Limit to top 60 for legible dendrogram
    if ret_matrix.shape[1] > 60:
        top_60 = profiles_sorted.head(60)
        top_60_syms = (top_60['symbol'].tolist() if 'symbol' in top_60.columns
                       else top_60.index.tolist())
        keep = [s for s in top_60_syms if s in ret_matrix.columns]
        ret_matrix = ret_matrix[keep]

    corr_matrix = ret_matrix.corr()
    # Distance = 1 - |correlation|
    dist_matrix = (1 - corr_matrix.abs()).clip(lower=0)
    np.fill_diagonal(dist_matrix.values, 0)
    condensed = squareform(dist_matrix.values, checks=False)
    linkage_matrix = linkage(condensed, method='ward')

    ax = axes[0]
    dendro = dendrogram(
        linkage_matrix,
        labels=list(ret_matrix.columns),
        leaf_rotation=90,
        leaf_font_size=7,
        color_threshold=0.7 * max(linkage_matrix[:, 2]),
        ax=ax,
    )
    ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage on Correlation Distance)', fontweight='bold')
    ax.set_ylabel('Distance (1 - |corr|)')
else:
    axes[0].text(0.5, 0.5, 'Insufficient data for dendrogram', ha='center', va='center', transform=axes[0].transAxes)

# ---- Chart 2: Cluster sizes bar chart ----
ax = axes[1]
cluster_ids = cluster_sizes.index.tolist()
sizes = cluster_sizes.values
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(cluster_ids))]
ax.bar(range(len(cluster_ids)), sizes, color=colors_bar, edgecolor='white')
ax.set_xticks(range(len(cluster_ids)))
ax.set_xticklabels([f'C{c}' for c in cluster_ids], fontsize=9)
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of Assets')
ax.set_title('Cluster Sizes (Crowded Sectors = Higher Bars)', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage6_clustering.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage6_clustering.png")

---
## Stage 7: Optimal Portfolio Selection (Final 10)

The core output: select 10 assets that maximise portfolio-level diversification, balancing risk-return quality, cluster representation, and liquidity. This uses the screener's optimisation engine which considers:
- Sharpe ratio (reward per unit risk)
- Cluster diversity (at most 1-2 per cluster)
- Tail dependence (avoid crash-correlated pairs)
- Idiosyncratic volatility (unique risk contribution)
- Liquidity (tradeable at scale)

In [ ]:
# ============================================================
# Cell 9: Stage 7 --- Optimal Selection (THE KEY OUTPUT)
# ============================================================

selected_assets = screener.select_optimal_portfolio(n_assets=None)

print("=" * 60)
print("  FINAL 10 SELECTED ASSETS")
print("=" * 60)
for i, asset in enumerate(selected_assets, 1):
    print(f"  {i:2d}. {asset}")
print("=" * 60)

# Get full profiles for the selected assets
sym_col = 'symbol' if 'symbol' in profiles_df.columns else None
if sym_col:
    selected_profiles = profiles_df[profiles_df[sym_col].isin(selected_assets)].copy()
else:
    selected_profiles = profiles_df.loc[profiles_df.index.isin(selected_assets)].copy()

print(f"\nFull profiles of selected assets:")
display(selected_profiles)

# ---- Radar / spider chart comparing the 10 assets on 6 dimensions ----
# Dimensions: return, volatility (inverted), Sharpe, max_dd (inverted), liquidity, uniqueness

radar_dims = []
dim_labels = []

# Normalise each metric to [0, 1] range for radar
def safe_normalize(series):
    s_min, s_max = series.min(), series.max()
    if s_max - s_min < 1e-12:
        return pd.Series(0.5, index=series.index)
    return (series - s_min) / (s_max - s_min)

if ret_col in selected_profiles.columns:
    radar_dims.append(safe_normalize(selected_profiles[ret_col]))
    dim_labels.append('Return')

if vol_col in selected_profiles.columns:
    radar_dims.append(1 - safe_normalize(selected_profiles[vol_col]))  # Lower vol = better
    dim_labels.append('Low Volatility')

if sharpe_col in selected_profiles.columns:
    radar_dims.append(safe_normalize(selected_profiles[sharpe_col]))
    dim_labels.append('Sharpe')

if dd_col in selected_profiles.columns:
    radar_dims.append(1 - safe_normalize(selected_profiles[dd_col].abs()))  # Lower DD = better
    dim_labels.append('Low Drawdown')

if 'volume_24h' in selected_profiles.columns:
    radar_dims.append(safe_normalize(np.log1p(selected_profiles['volume_24h'])))
    dim_labels.append('Liquidity')
elif 'avg_daily_volume' in selected_profiles.columns:
    radar_dims.append(safe_normalize(np.log1p(selected_profiles['avg_daily_volume'])))
    dim_labels.append('Liquidity')

if 'idio_vol' in selected_profiles.columns:
    radar_dims.append(safe_normalize(selected_profiles['idio_vol']))
    dim_labels.append('Uniqueness')

n_dims = len(dim_labels)

if n_dims >= 3:
    angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
    angles += angles[:1]  # Close the polygon

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

    asset_labels = (selected_profiles['symbol'].tolist() if 'symbol' in selected_profiles.columns
                    else selected_profiles.index.tolist())

    for idx in range(len(selected_profiles)):
        values = [float(radar_dims[d].iloc[idx]) for d in range(n_dims)]
        values += values[:1]  # Close the polygon
        color = PALETTE[idx % len(PALETTE)]
        ax.plot(angles, values, 'o-', linewidth=1.5, color=color, label=asset_labels[idx], markersize=4)
        ax.fill(angles, values, alpha=0.05, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dim_labels, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title('Final 10 Assets: Multi-Dimensional Comparison', fontweight='bold', pad=20, fontsize=14)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/stage7_radar_chart.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {SAVE_DIR}/stage7_radar_chart.png")
else:
    print("Insufficient dimensions for radar chart.")

# ---- Asset class breakdown ----
if 'asset_class' in selected_profiles.columns:
    print(f"\nAsset class breakdown:")
    class_counts = selected_profiles['asset_class'].value_counts()
    for cls, count in class_counts.items():
        members = selected_profiles[selected_profiles['asset_class'] == cls]
        syms = members['symbol'].tolist() if 'symbol' in members.columns else members.index.tolist()
        print(f"  {cls}: {count} --- {', '.join(str(s) for s in syms)}")

---
## Stage 8: Validation

Compare the optimised selection against naive alternatives:
- **Top 10 by market cap** (capitalization-weighted baseline)
- **Top 10 by Sharpe** (pure performance chasing)
- **Random 10** (null hypothesis)

Metrics: diversification ratio, portfolio Sharpe, CVaR, spanning test.

In [ ]:
# ============================================================
# Cell 10: Stage 8 --- Validation
# ============================================================

validation_results = screener.validate_selection(selected=selected_assets)

print("Validation Results:")
print("=" * 60)

if isinstance(validation_results, dict):
    for key, value in validation_results.items():
        if isinstance(value, (int, float, np.floating)):
            print(f"  {key}: {value:.4f}")
        elif isinstance(value, dict):
            print(f"  {key}:")
            for k2, v2 in value.items():
                if isinstance(v2, (int, float, np.floating)):
                    print(f"    {k2}: {v2:.4f}")
                else:
                    print(f"    {k2}: {v2}")
        else:
            print(f"  {key}: {value}")

# ---- Build comparison portfolios ----
# Top 10 by market cap
mcap_col = 'market_cap' if 'market_cap' in universe_df.columns else None
if mcap_col:
    top10_mcap = universe_df.nlargest(10, mcap_col)
    top10_mcap_syms = (top10_mcap['symbol'].tolist() if 'symbol' in top10_mcap.columns
                       else top10_mcap.index.tolist())
else:
    top10_mcap_syms = universe_df.head(10)['symbol'].tolist() if 'symbol' in universe_df.columns else universe_df.head(10).index.tolist()

# Top 10 by Sharpe
top10_sharpe = profiles_sorted.head(10)
top10_sharpe_syms = (top10_sharpe['symbol'].tolist() if 'symbol' in top10_sharpe.columns
                     else top10_sharpe.index.tolist())

# Random 10
np.random.seed(42)
all_stat_syms = stat_quality_assets.copy()
np.random.shuffle(all_stat_syms)
random10_syms = all_stat_syms[:10]

# ---- Compute portfolio metrics for each selection ----
def compute_portfolio_metrics(asset_list, label):
    """Compute equal-weight portfolio metrics for a list of assets."""
    available = [s for s in asset_list if s in ohlcv_data and len(ohlcv_data[s]) > 60]
    if len(available) < 2:
        return {'method': label, 'n_assets': len(available),
                'diversification_ratio': np.nan, 'sharpe': np.nan, 'cvar_5pct': np.nan}

    rets = {}
    for sym in available:
        r = ohlcv_data[sym]['close'].pct_change().dropna()
        rets[sym] = r
    ret_df = pd.DataFrame(rets).dropna(how='all').fillna(0)

    n = ret_df.shape[1]
    w = np.ones(n) / n
    cov = ret_df.cov().values
    port_vol = np.sqrt(w @ cov @ w)
    asset_vols = np.sqrt(np.maximum(np.diag(cov), 1e-12))
    div_ratio = float((asset_vols @ w) / port_vol) if port_vol > 0 else 1.0

    port_ret = (ret_df.values @ w)
    ann_factor = np.sqrt(365)  # daily data
    mean_ret = port_ret.mean() * 365
    sharpe = mean_ret / (port_vol * ann_factor) if port_vol > 0 else 0.0

    # CVaR 5%
    sorted_rets = np.sort(port_ret)
    cutoff = int(len(sorted_rets) * 0.05)
    cvar = float(sorted_rets[:max(cutoff, 1)].mean()) if len(sorted_rets) > 0 else 0.0

    return {
        'method': label,
        'n_assets': n,
        'diversification_ratio': round(div_ratio, 4),
        'sharpe': round(sharpe, 4),
        'cvar_5pct': round(cvar, 6),
    }

comparison_results = [
    compute_portfolio_metrics(selected_assets, 'Optimised (Ours)'),
    compute_portfolio_metrics(top10_mcap_syms, 'Top 10 Market Cap'),
    compute_portfolio_metrics(top10_sharpe_syms, 'Top 10 Sharpe'),
    compute_portfolio_metrics(random10_syms, 'Random 10'),
]

comparison_df = pd.DataFrame(comparison_results)
print("\nPortfolio Comparison:")
display(comparison_df)

# ---- Chart 1: Grouped bar chart ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

methods = comparison_df['method'].tolist()
x = np.arange(len(methods))
bar_width = 0.25
method_colors = [PALETTE[0], PALETTE[3], PALETTE[5], PALETTE[7]]

# (a) Diversification ratio and Sharpe
ax = axes[0]
bars1 = ax.bar(x - bar_width/2, comparison_df['diversification_ratio'], bar_width,
               label='Diversification Ratio', color=PALETTE[0], edgecolor='white')
bars2 = ax.bar(x + bar_width/2, comparison_df['sharpe'], bar_width,
               label='Sharpe Ratio', color=PALETTE[2], edgecolor='white')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if not np.isnan(height):
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    if not np.isnan(height):
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=10, rotation=15, ha='right')
ax.set_title('Diversification Ratio & Sharpe by Method', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylabel('Ratio')
sns.despine(ax=ax)

# (b) Correlation heatmap of the final 10
ax = axes[1]
selected_avail = [s for s in selected_assets if s in ohlcv_data and len(ohlcv_data[s]) > 60]
if len(selected_avail) >= 2:
    sel_rets = {}
    for sym in selected_avail:
        r = ohlcv_data[sym]['close'].pct_change().dropna()
        sel_rets[sym] = r
    sel_ret_df = pd.DataFrame(sel_rets).dropna(how='all').fillna(0)
    sel_corr = sel_ret_df.corr()

    mask = np.triu(np.ones_like(sel_corr, dtype=bool), k=1)
    sns.heatmap(
        sel_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
        center=0, vmin=-1, vmax=1, square=True, ax=ax,
        linewidths=0.5, cbar_kws={'shrink': 0.8},
    )
    ax.set_title('Correlation Matrix: Final 10', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Insufficient data for correlation heatmap',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/stage8_validation.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {SAVE_DIR}/stage8_validation.png")

# ---- Spanning test summary ----
if 'spanning_test' in validation_results:
    st = validation_results['spanning_test']
    print(f"\nSpanning Test:")
    print(f"  F-statistic: {st.get('test_statistic', 'N/A')}")
    print(f"  p-value:     {st.get('p_value', 'N/A')}")
    print(f"  Result:      {st.get('interpretation', 'N/A')}")

---
## Stage 9: Asset Classification

Classify each selected asset by its economic role in the portfolio. This is the table that goes directly into the report's asset tokenisation section.

In [ ]:
# ============================================================
# Cell 11: Asset Classification
# ============================================================

classification = screener.classify_assets(selected=selected_assets)

print("Asset Classification for Report")
print("=" * 70)

# Build a clean display table
class_rows = []
for asset_class, assets_in_class in classification.items():
    for asset in assets_in_class:
        # Try to get profile info
        profile_row = None
        if sym_col and sym_col in profiles_df.columns:
            matches = profiles_df[profiles_df[sym_col] == asset]
            if len(matches) > 0:
                profile_row = matches.iloc[0]
        elif asset in profiles_df.index:
            profile_row = profiles_df.loc[asset]

        row = {
            'Asset': asset,
            'Class': asset_class,
        }

        # Determine value source and role
        class_lower = asset_class.lower()
        if 'crypto' in class_lower or 'layer' in class_lower or 'l1' in class_lower:
            row['Value Source'] = 'Network utility + adoption'
            row['Role in Portfolio'] = 'Growth / beta exposure'
        elif 'stak' in class_lower or 'yield' in class_lower or 'lst' in class_lower:
            row['Value Source'] = 'Staking yield + underlying asset'
            row['Role in Portfolio'] = 'Income generation + moderate risk'
        elif 'treasur' in class_lower or 'rwa' in class_lower:
            row['Value Source'] = 'Tokenised real-world assets (gov bonds)'
            row['Role in Portfolio'] = 'Stability + yield floor'
        elif 'stable' in class_lower:
            row['Value Source'] = 'USD peg (reserves-backed)'
            row['Role in Portfolio'] = 'Cash buffer + rebalancing liquidity'
        elif 'defi' in class_lower:
            row['Value Source'] = 'Protocol revenue + governance'
            row['Role in Portfolio'] = 'Idiosyncratic alpha + low correlation'
        else:
            row['Value Source'] = 'Market-driven'
            row['Role in Portfolio'] = 'Diversification'

        if profile_row is not None:
            if sharpe_col in profile_row.index:
                row['Sharpe'] = f"{profile_row[sharpe_col]:.2f}"
            if vol_col in profile_row.index:
                row['Ann. Vol'] = f"{profile_row[vol_col]*100:.1f}%"

        class_rows.append(row)

class_table = pd.DataFrame(class_rows)
display(class_table)

# Save classification
class_table.to_csv(f"{SAVE_DIR}/asset_classification.csv", index=False)
print(f"\nSaved: {SAVE_DIR}/asset_classification.csv")

---
## Summary and Export

Save all results and the final asset list for downstream pipeline consumption.

In [ ]:
# ============================================================
# Cell 12: Summary & Export
# ============================================================

import json

print("=" * 60)
print("  UNIVERSE SCREENING COMPLETE")
print("=" * 60)
print(f"\n  Pipeline: 500 -> {post_liquidity_count} (liquidity) -> {post_stat_count} (stats) -> 10 (optimised)")
print(f"\n  Final 10 Assets:")
for i, asset in enumerate(selected_assets, 1):
    print(f"    {i:2d}. {asset}")

# ---- Save selected assets ----
os.makedirs(f"{SAVE_DIR}", exist_ok=True)

results_payload = {
    'selected_assets': selected_assets,
    'screening_summary': {
        'initial_universe': pre_liquidity_count,
        'after_liquidity': post_liquidity_count,
        'after_statistical_quality': post_stat_count,
        'final_selection': len(selected_assets),
    },
    'classification': classification,
    'comparison': comparison_df.to_dict(orient='records'),
}

with open(f"{SAVE_DIR}/selected_assets.json", 'w') as f:
    json.dump(results_payload, f, indent=2, default=str)

print(f"\n  Results saved to:")
print(f"    {SAVE_DIR}/selected_assets.json")
print(f"    {SAVE_DIR}/asset_classification.csv")
print(f"    {SAVE_DIR}/stage1_universe_distributions.png")
print(f"    {SAVE_DIR}/stage2_data_availability.png")
print(f"    {SAVE_DIR}/stage3_liquidity_funnel.png")
print(f"    {SAVE_DIR}/stage4_statistical_quality.png")
print(f"    {SAVE_DIR}/stage5_risk_return_profiles.png")
print(f"    {SAVE_DIR}/stage6_clustering.png")
print(f"    {SAVE_DIR}/stage7_radar_chart.png")
print(f"    {SAVE_DIR}/stage8_validation.png")

# Save profiles for the selected assets
selected_profiles.to_csv(f"{SAVE_DIR}/selected_profiles.csv", index=True)
print(f"    {SAVE_DIR}/selected_profiles.csv")

print("\n" + "=" * 60)
print("  NEXT STEPS:")
print("  1. Update config.yaml with these assets")
print("  2. Run train_sac.ipynb to train the RL agent")
print("  3. Run the full ML pipeline")
print("=" * 60)

---
## Update Config (Optional)

Preview what `config.yaml` would look like with the new 10-asset selection. Review before writing -- this cell does NOT auto-write.

In [ ]:
# ============================================================
# Cell 13: Update Config (Optional --- review before running)
# ============================================================

# Build the updated config preview
new_config = config.copy()
new_config['data'] = config['data'].copy()
new_config['data']['assets'] = selected_assets

# Update RL action dimension to match new asset count
new_config['rl'] = config['rl'].copy()
new_config['rl']['action_dim'] = len(selected_assets)

# Update ensemble constraints
new_config['ensemble'] = config['ensemble'].copy()
new_config['ensemble']['max_single_asset'] = round(min(0.40, 3.0 / len(selected_assets)), 2)
new_config['ensemble']['min_single_asset'] = round(max(0.02, 0.5 / len(selected_assets)), 2)

print("Preview of updated config.yaml:")
print("=" * 60)
print(yaml.dump(new_config, default_flow_style=False, sort_keys=False))
print("=" * 60)

# ---- Uncomment the lines below to WRITE the config ----
# CAUTION: This overwrites your existing config.yaml!
#
# with open("config.yaml", 'w') as f:
#     yaml.dump(new_config, f, default_flow_style=False, sort_keys=False)
# print("config.yaml updated with new asset selection.")
#
# ---- End optional write ----

print("\nTo apply: uncomment the write block above and re-run this cell.")
print("Then proceed to train_sac.ipynb.")